In [ ]:
import weaviate
from weaviate.classes.config import Configure
import pickle , json
import numpy as np
import os
from tqdm import tqdm
from dotenv import load_dotenv


load_dotenv()

True

In [2]:
# Best practice: store your credentials in environment variables
weaviate_url = os.environ["WEAVIATE_URL"]
weaviate_api_key = os.environ["WEAVIATE_API_KEY"]

In [ ]:
# Step 1.1: Connect to your Weaviate Cloud instance
with weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=weaviate_api_key,
) as client:
    
    # Step 1.2: Create a collection
    movies = client.collections.create(
        name="Movie",
        vector_config=Configure.Vectors.self_provided(),  # No automatic vectorization since we're providing vectors
    )



    # Step 1.3: Import three objects
    data_objects = [
        {"properties": {"title": "The Matrix", "description": "A computer hacker learns about the true nature of reality and his role in the war against its controllers.", "genre": "Science Fiction"},
        "vector": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]},
        {"properties": {"title": "Spirited Away", "description": "A young girl becomes trapped in a mysterious world of spirits and must find a way to save her parents and return home.", "genre": "Animation"},
        "vector": [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]},
        {"properties": {"title": "The Lord of the Rings: The Fellowship of the Ring", "description": "A meek Hobbit and his companions set out on a perilous journey to destroy a powerful ring and save Middle-earth.", "genre": "Fantasy"},
        "vector": [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]}
    ]

    # Insert the objects with vectors
    movies = client.collections.get("Movie")
    with movies.batch.fixed_size(batch_size=200) as batch:
        for obj in data_objects:
            batch.add_object(properties=obj["properties"], vector=obj["vector"])

    print(
        f"Imported {len(data_objects)} objects with vectors into the Movie collection"
    )

In [3]:

# Step 2.1: Connect to your Weaviate Cloud instance
with weaviate.connect_to_weaviate_cloud(
    cluster_url=weaviate_url,
    auth_credentials=weaviate_api_key,
) as client:

    # Step 2.2: Use this collection
    movies = client.collections.use("Movie")

    # Step 2.3: Perform a vector search with NearVector
    response = movies.query.near_vector(
        near_vector=[0.11, 0.21, 0.31, 0.41, 0.51, 0.61, 0.71, 0.81], 
        limit=2
    )

    for obj in response.objects:
        print(json.dumps(obj.properties, indent=2))  # Inspect the results

{
  "genre": "Science Fiction",
  "description": "A computer hacker learns about the true nature of reality and his role in the war against its controllers.",
  "title": "The Matrix"
}
{
  "genre": "Animation",
  "title": "Spirited Away",
  "description": "A young girl becomes trapped in a mysterious world of spirits and must find a way to save her parents and return home."
}


In [4]:
with open('dataset/metadata_chunks.pkl','rb') as m:
    metadata_chunks = pickle.load(m)

In [8]:
metadata_chunks[:2]

[{'celex': '32019D0276',
  'act_name': 'Decision (EU) 2019/276 of the European Parliament and of the Council of 12 December 2018 on the mobilisation of the Flexibility Instrument to reinforce key programmes for the competitiveness of the EU and to finance immediate budgetary measures to address the ongoing challenges of migration, refugee inflows and security threats',
  'act_type': 'Decision',
  'eurovoc': 'aid to refugees; budget appropriation; EC general budget; European security; refugee; migratory flow; Community financing; Community migration policy; Community expenditure; commitment appropriation',
  'subject_matter': 'cooperation policy;  budget;  EU finance;  international security;  migration',
  'legal_basis': '32013Q1220(01)',
  'date': '2018-12-12',
  'authors': 'European Parliament; European Council',
  'status': 'In Force',
  'cites': '32013R1311',
  'treaty': 'TFEU',
  'additional_info': 'No additional information',
  'chunk_number': 1,
  'total_chunks': 2,
  'document_

In [6]:
metordata= metadata_chunks[50000]
for key , value in metordata.items():
    print(f"{key}: {value}")


celex: 32011D0890
act_name: 2011/890/EU: Commission Implementing Decision of 22Ã December 2011 providing the rules for the establishment, the management and the functioning of the network of national responsible authorities on eHealth
act_type: Decision_IMPL
eurovoc: Euronet; automatic information system; health; exchange of information; digital public service
subject_matter: communications;  information and information processing;  health;  executive power and public service
legal_basis: 32011L0024
date: 2011-12-22
authors: European Commission
status: In Force
cites: 32006D1639; 32002L58; 32001D844; 32007D1350; 31995L46
treaty: TFEU (2008)
additional_info: No additional information
chunk_number: 1
total_chunks: 4
document_length: 8293


In [9]:
with open('dataset/text_chunks.pkl','rb') as c:
    text_chunks = pickle.load(c)

In [5]:
for index , chunk in enumerate(text_chunks[:20]):
    print(f"index : {index}")
    print(chunk)

index : 0
22.2.2019 EN Official Journal of the European Union L 54/3 DECISION (EU) 2019/276 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 12 December 2018 on the mobilisation of the Flexibility Instrument to reinforce key programmes for the competitiveness of the EU and to finance immediate budgetary measures to address the ongoing challenges of migration, refugee inflows and security threats THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, Having regard to the Interinstitutional Agreement of 2 December 2013 between the European Parliament, the Council and the Commission on budgetary discipline, on cooperation in budgetary matters and on sound financial management (1), and in particular point 12 thereof, Having regard to the proposal from the European Commission, Whereas, (1) The Flexibility Instrument is intended to allow the financing of clearly identified expenditure which could not be financed wi

In [10]:
with open('dataset/embeddings.npy','rb') as e:
    emmbedings = np.load(e)    

In [ ]:
type(emmbedings)

numpy.ndarray

In [1]:
print("live")

live


In [7]:
print(f" this is len meta: {len(metadata_chunks)}")
print(f" this is len chunks: {len(text_chunks)}")
print(f" this is len embeddings: {len(emmbedings)}")

 this is len meta: 694515
 this is len chunks: 694515
 this is len embeddings: 694515


# locallll

In [ ]:
!docker-compose up -d

In [8]:
import weaviate
from weaviate.classes.config import Configure

# Step 1.1: Connect to your local Weaviate instance
with weaviate.connect_to_local() as client:

    # Step 1.2: Create a collection
    movies = client.collections.create(
        name="Movie",
        vector_config=Configure.Vectors.self_provided(),  # No automatic vectorization since we're providing vectors
    )

    # Step 1.3: Import three objects
    data_objects = [
        {"properties": {"title": "The Matrix", "description": "A computer hacker learns about the true nature of reality and his role in the war against its controllers.", "genre": "Science Fiction"},
        "vector": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]},
        {"properties": {"title": "Spirited Away", "description": "A young girl becomes trapped in a mysterious world of spirits and must find a way to save her parents and return home.", "genre": "Animation"},
        "vector": [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]},
        {"properties": {"title": "The Lord of the Rings: The Fellowship of the Ring", "description": "A meek Hobbit and his companions set out on a perilous journey to destroy a powerful ring and save Middle-earth.", "genre": "Fantasy"},
        "vector": [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]}
    ]

    # Insert the objects with vectors
    movies = client.collections.get("Movie")
    with movies.batch.fixed_size(batch_size=200) as batch:
        for obj in data_objects:
            batch.add_object(properties=obj["properties"], vector=obj["vector"])

    print(
        f"Imported {len(data_objects)} objects with vectors into the Movie collection"
    )

Imported 3 objects with vectors into the Movie collection


In [9]:
import weaviate
import json

# Step 2.1: Connect to your local Weaviate instance
with weaviate.connect_to_local() as client:

    # Step 2.2: Use this collection
    movies = client.collections.use("Movie")

    # Step 2.3: Perform a vector search with NearVector
    response = movies.query.near_vector(
        near_vector=[0.11, 0.21, 0.31, 0.41, 0.51, 0.61, 0.71, 0.81], 
        limit=2
    )

    for obj in response.objects:
        print(json.dumps(obj.properties, indent=2))  # Inspect the results

{
  "title": "The Matrix",
  "description": "A computer hacker learns about the true nature of reality and his role in the war against its controllers.",
  "genre": "Science Fiction"
}
{
  "title": "Spirited Away",
  "description": "A young girl becomes trapped in a mysterious world of spirits and must find a way to save her parents and return home.",
  "genre": "Animation"
}


# OUR RAG

In [2]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType

In [15]:
with weaviate.connect_to_local() as client:

    # Step 1.2: Create a collection
    eur_laws = client.collections.create(
        name="Euro_Laws",
        vector_config=Configure.Vectors.self_provided(),    
        properties=[
            Property(name="text", data_type=DataType.TEXT),
            Property(name="celex", data_type=DataType.TEXT),
            Property(name="act_name", data_type=DataType.TEXT),
            Property(name="act_type", data_type=DataType.TEXT),
            Property(name="eurovoc", data_type=DataType.TEXT),
            Property(name="subject_matter", data_type=DataType.TEXT),
            Property(name="legal_basis", data_type=DataType.TEXT),
            Property(name="authors", data_type=DataType.TEXT),
            Property(name="status", data_type=DataType.TEXT),
            Property(name="cites", data_type=DataType.TEXT),
            Property(name="treaty", data_type=DataType.TEXT),
            Property(name="additional_info", data_type=DataType.TEXT),
            Property(name="chunk_number", data_type=DataType.INT),
            Property(name="total_chunks", data_type=DataType.INT),
            Property(name="document_length", data_type=DataType.INT),
        ],
    )



In [ ]:
with weaviate.connect_to_local() as client:        
    # Batch insert
    eur_laws = client.collections.get("Euro_Laws")

    with eur_laws.batch.fixed_size(batch_size=256) as batch:
        for i in tqdm(range(len(text_chunks))):
            text = text_chunks[i]
            meta = metadata_chunks[i]
            vector = emmbedings[i].tolist()

            properties = {
                "text": text,
                "celex": str(meta.get("celex", "")),
                "act_name": str(meta.get("act_name", "")),
                "act_type": str(meta.get("act_type", "")), 
                "eurovoc": str(meta.get("eurovoc", "")),
                "subject_matter": str(meta.get("subject_matter", "")),
                "legal_basis": str(meta.get("legal_basis", "")),
                "authors": str(meta.get("authors", "")),
                "status": str(meta.get("status", "")),
                "cites": str(meta.get("cites", "")),
                "treaty": str(meta.get("treaty", "")),
                "additional_info": str(meta.get("additional_info", "")),
                "chunk_number": int(meta.get("chunk_number", 0)),
                "total_chunks": int(meta.get("total_chunks", 0)),
                "document_length": int(meta.get("document_length", 0)),
            }

            batch.add_object(
                properties=properties,
                vector=vector
            )

 73%|███████▎  | 506880/694515 [12:02<06:34, 475.84it/s]  {'message': 'Failed to send all objects in a batch of 256', 'error': 'WeaviateInsertManyAllFailedError(\'Every object failed during insertion. Here is the set of all errors: resolve node name "node1" to host\')'}
{'message': 'Failed to send 256 objects in a batch of 256. Please inspect client.batch.failed_objects or collection.batch.failed_objects for the failed objects.'}
{'message': 'Failed to send all objects in a batch of 256', 'error': 'WeaviateInsertManyAllFailedError(\'Every object failed during insertion. Here is the set of all errors: resolve node name "node1" to host\')'}
{'message': 'Failed to send 256 objects in a batch of 256. Please inspect client.batch.failed_objects or collection.batch.failed_objects for the failed objects.'}
 73%|███████▎  | 507556/694515 [12:05<08:03, 386.46it/s]{'message': 'Failed to send all objects in a batch of 256', 'error': 'WeaviateInsertManyAllFailedError(\'Every object failed during in

# SentenceTransformer

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
embedding_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

query_embedding = embedding_model.encode([enhanced_query])


In [12]:
vector = [0.056547436863183975,
  0.06741845607757568,
  0.027699511498212814,
  -0.0007179611129686236,
  -0.04737325385212898,
  0.03709559142589569,
  -0.01961769349873066,
  -0.02652956359088421,
  -0.05675952509045601,
  -0.00891951285302639,
  -0.037390656769275665,
  0.049838945269584656,
  0.019901545718312263,
  -0.004376955330371857,
  0.012217922136187553,
  0.019090881571173668,
  0.04277316853404045,
  -0.014752276241779327,
  0.009167377837002277,
  -0.014557000249624252,
  0.04466339945793152,
  0.04789062589406967,
  -0.02886529080569744,
  -0.003770110197365284,
  -0.016385991126298904,
  0.02320822887122631,
  -0.015027343295514584,
  -0.035184092819690704,
  -0.026811733841896057,
  -0.011842114850878716,
  -0.06077006831765175,
  0.0028461499605327845,
  0.009521739557385445,
  0.019169265404343605,
  1.333587078988785e-06,
  -0.022509852424263954,
  -0.023466281592845917,
  -0.028328249230980873,
  -0.05665050446987152,
  0.009548759087920189,
  -0.03175557777285576,
  -0.04086127132177353,
  0.03614949807524681,
  0.008246798068284988,
  -0.04337642714381218,
  -0.03678591921925545,
  0.01385288592427969,
  0.06283190101385117,
  -0.017166273668408394,
  -0.013511226512491703,
  0.02623242512345314,
  -0.08397681266069412,
  -0.0386267825961113,
  -0.04932752251625061,
  -0.026541022583842278,
  0.026032395660877228,
  -0.016618670895695686,
  0.024935558438301086,
  0.010971298441290855,
  0.017951827496290207,
  0.007330111227929592,
  -0.008917190134525299,
  0.018750900402665138,
  -0.018827183172106743,
  -0.032355185598134995,
  -0.04362019523978233,
  -0.08938635140657425,
  0.016311252489686012,
  0.011327733285725117,
  -0.016156848520040512,
  -0.06346817314624786,
  0.05634811520576477,
  -0.007672521285712719,
  0.031895168125629425,
  0.0014868342550471425,
  -0.07575228810310364,
  0.025180255994200706,
  0.03689469397068024,
  -0.01923709735274315,
  0.011398748494684696,
  -0.0018596850568428636,
  0.07896733283996582,
  -0.011846066452562809,
  0.037953317165374756,
  0.008982906118035316,
  -0.05405356362462044,
  0.017407264560461044,
  -0.02055215835571289,
  0.023336708545684814,
  -3.102875780314207e-05,
  -0.12327086925506592,
  0.04915912449359894,
  -0.02889491803944111,
  -0.02146732248365879,
  -0.03666761517524719,
  -0.014189640060067177,
  -0.002564468188211322,
  -0.02208275906741619,
  0.02732572890818119,
  -0.026430919766426086,
  -0.008463599719107151,
  -0.019688105210661888,
  0.027425363659858704,
  -0.004871037323027849,
  0.0022475288715213537,
  -0.0405048206448555,
  -0.0452428013086319,
  -0.004625077359378338,
  -0.06326557695865631,
  0.02754835970699787,
  0.005444996990263462,
  -0.018476296216249466,
  0.00815139152109623,
  -0.009929404594004154,
  0.059138234704732895,
  0.05608992278575897,
  -0.029208561405539513,
  -0.015210956335067749,
  0.0006001541623845696,
  -0.008715075440704823,
  -0.05407378077507019,
  -0.025219833478331566,
  -0.05188538506627083,
  0.004204102326184511,
  0.03835252299904823,
  -0.09075403958559036,
  -0.039881862699985504,
  0.02685123309493065,
  0.007130127400159836,
  0.011870888993144035,
  -0.030074289068579674,
  0.014896073378622532,
  -0.0444067157804966,
  -0.014429098926484585,
  0.028134429827332497,
  0.07419641315937042,
  -0.033532459288835526,
  0.014444897882640362,
  -0.027439648285508156,
  -0.015636079013347626,
  -0.019378334283828735,
  0.0014484265120700002,
  0.01817707158625126,
  -0.03671404719352722,
  -0.029753152281045914,
  -0.059408124536275864,
  -0.021910086274147034,
  0.0654207244515419,
  -0.009283174760639668,
  0.010041195899248123,
  -0.0006125241634435952,
  -0.0002935952215921134,
  0.017652174457907677,
  -0.050472039729356766,
  -0.01797269843518734,
  -0.0158503707498312,
  -0.031831465661525726,
  -0.008315741084516048,
  0.018362129107117653,
  0.004862227942794561,
  0.04260963574051857,
  0.07037353515625,
  0.03831145912408829,
  0.023419160395860672,
  -0.016225120052695274,
  -0.0023014203179627657,
  0.0724899098277092,
  -0.004728846717625856,
  0.045106131583452225,
  0.00020771690469700843,
  0.02901601605117321,
  -0.04147667810320854,
  -0.00483434135094285,
  0.006770616862922907,
  -0.009115802124142647,
  0.09628411382436752,
  -0.024822039529681206,
  0.0010133328614756465,
  0.011842777021229267,
  0.011025439947843552,
  -0.036275461316108704,
  -0.032398514449596405,
  -0.022738883271813393,
  0.015049128793179989,
  -0.029421864077448845,
  0.08012360334396362,
  0.0037910789251327515,
  -0.0208795964717865,
  -0.019667338579893112,
  0.04254637658596039,
  0.04192343354225159,
  -0.029773496091365814,
  -0.013545757159590721,
  0.006686931475996971,
  -0.0017090246547013521,
  -0.00108528439886868,
  0.0358491949737072,
  -0.045323893427848816,
  0.011253194883465767,
  -0.002675369381904602,
  0.006911060307174921,
  0.026570461690425873,
  -0.010377860628068447,
  0.002967468462884426,
  -0.008498064242303371,
  0.016302358359098434,
  0.03504214435815811,
  -0.022314993664622307,
  -0.00016161854728125036,
  0.008533983491361141,
  0.00772194704040885,
  0.026616692543029785,
  0.022080745548009872,
  -0.018479373306035995,
  -0.021243220195174217,
  0.045577630400657654,
  -0.01790354959666729,
  -0.03163272142410278,
  0.006639854051172733,
  -0.04508485645055771,
  -0.016498910263180733,
  -0.04343313351273537,
  0.05443587526679039,
  -0.009826614521443844,
  -0.04849575087428093,
  -0.03152471035718918,
  -0.0005540716811083257,
  0.010406244546175003,
  -0.014437079429626465,
  -0.024697838351130486,
  0.01115002203732729,
  -0.01991397887468338,
  0.015301695093512535,
  -0.015479535795748234,
  0.03851369023323059,
  0.05155683308839798,
  -0.05014394596219063,
  -0.09040310978889465,
  -0.02125122956931591,
  -0.038466427475214005,
  -0.006615763995796442,
  0.03663618117570877,
  -0.02526676654815674,
  -0.01241480652242899,
  -0.015793854370713234,
  -0.011884650215506554,
  -0.026368597522377968,
  -0.05186138674616814,
  0.018891558051109314,
  -0.0014487260486930609,
  0.00808935146778822,
  0.012974430806934834,
  0.02372029609978199,
  -0.0011721288319677114,
  0.03551115468144417,
  -0.11333879828453064,
  0.029142484068870544,
  0.028452666476368904,
  0.012942624278366566,
  -0.04973751679062843,
  -0.053320903331041336,
  0.024654332548379898,
  -0.023778367787599564,
  0.03394123911857605,
  0.05013739690184593,
  -0.004704222083091736,
  -0.012789406813681126,
  -0.019960014149546623,
  -0.0003050389641430229,
  0.02431480400264263,
  -0.0343799814581871,
  0.0009662633528932929,
  0.06447842717170715,
  -0.004999944940209389,
  -0.031447723507881165,
  -0.0343443937599659,
  0.03770386800169945,
  0.05821290239691734,
  -0.009894887916743755,
  -0.013732517138123512,
  0.014884489588439465,
  -0.04633234813809395,
  -0.002010093303397298,
  4.6685967390658334e-05,
  0.028864556923508644,
  0.0112676452845335,
  -0.018208308145403862,
  0.004549429286271334,
  0.020417822524905205,
  -0.04048437252640724,
  0.03115513175725937,
  0.01832456700503826,
  -0.03182828053832054,
  0.04380311071872711,
  0.019499637186527252,
  -0.03600826486945152,
  -0.06578333675861359,
  0.0006945643108338118,
  -0.018841074779629707,
  0.014910893514752388,
  0.052378956228494644,
  -0.02628217451274395,
  -0.0023832933511584997,
  -0.017537401989102364,
  -0.0005525774322450161,
  -0.03464173898100853,
  0.015599243342876434,
  0.020795084536075592,
  -0.09488295763731003,
  -0.023167796432971954,
  0.0365726463496685,
  0.030020007863640785,
  0.0057506621815264225,
  -0.04944290220737457,
  0.016152404248714447,
  -0.03458870202302933,
  -0.027964716777205467,
  -0.0009529629023745656,
  -0.026376286521553993,
  -0.027507798746228218,
  0.02878992259502411,
  0.02777068503201008,
  0.05671058967709541,
  0.011023415252566338,
  0.009118851274251938,
  0.05724076181650162,
  0.031396325677633286,
  0.001881975680589676,
  0.07718536257743835,
  -0.007370410952717066,
  -0.017253028228878975,
  0.047025442123413086,
  -0.024494268000125885,
  0.06725002080202103,
  0.031103499233722687,
  0.0005552068469114602,
  0.03205753117799759,
  0.024839960038661957,
  0.025626961141824722,
  0.0066324337385594845,
  0.0014622806338593364,
  -0.01204547006636858,
  0.04298200085759163,
  -0.11700604110956192,
  0.003849724540486932,
  0.0533432699739933,
  0.02258409559726715,
  -0.005617227405309677,
  -0.04644695296883583,
  -0.02591240033507347,
  -0.025879787281155586,
  -0.01760980673134327,
  -0.038432035595178604,
  -0.023356173187494278,
  -0.04205723851919174,
  0.0020040059462189674,
  0.011963092721998692,
  -0.026627808809280396,
  -0.06263723969459534,
  -0.023324737325310707,
  -0.02310032770037651,
  -0.002289858181029558,
  -0.06334278732538223,
  0.0752272829413414,
  0.02997489459812641,
  0.08442170917987823,
  0.056775979697704315,
  -0.004156581591814756,
  -0.039130110293626785,
  -0.03485684096813202,
  -0.07265046238899231,
  0.027259206399321556,
  -0.05964342877268791,
  0.034579891711473465,
  -0.0050811199471354485,
  -0.002539312466979027,
  0.009335865266621113,
  -0.018365375697612762,
  0.03589232638478279,
  -0.030878199264407158,
  0.017827551811933517,
  -0.09579376131296158,
  -0.004724054131656885,
  -0.03477691859006882,
  0.022565890103578568,
  0.02567802183330059,
  -0.013195786625146866,
  -0.05103566497564316,
  -0.02972080372273922,
  0.06179051846265793,
  0.025795677676796913,
  -0.023404857143759727,
  0.012900426983833313,
  -0.0435112826526165,
  0.037877365946769714,
  -0.03952058404684067,
  -0.055598802864551544,
  0.0357343815267086,
  -0.016106432303786278,
  0.004204514902085066,
  0.00027824213611893356,
  -0.013519754633307457,
  0.07683563977479935,
  0.06159420311450958,
  -0.03494745120406151,
  -0.015908900648355484,
  0.04747651889920235,
  -0.046116527169942856,
  -0.03020642139017582,
  0.05003904178738594,
  -0.08162591606378555,
  0.06794428825378418,
  -0.00013951858272776008,
  0.048478029668331146,
  0.010309071280062199,
  -0.018023531883955002,
  0.0009647586266510189,
  -0.03813246265053749,
  0.02310088835656643,
  0.03345169126987457,
  -0.065678671002388,
  -0.006345003377646208,
  0.004955202806740999,
  -0.0013086263788864017,
  0.06573880463838577,
  0.013144202530384064,
  -0.03520508110523224,
  -0.018908943980932236,
  0.00016003218479454517,
  -0.016161831095814705,
  0.047920022159814835,
  0.0066132927313447,
  0.021724261343479156,
  -0.13603028655052185,
  -0.04188130050897598,
  0.0416247732937336,
  -0.06542897969484329,
  0.0329512320458889,
  -0.033288486301898956,
  0.044494081288576126,
  -0.032627563923597336,
  0.03430754691362381,
  0.026007259264588356,
  -0.00277238292619586,
  -0.011957473121583462,
  0.0706852525472641,
  0.007240160834044218,
  -0.07238709926605225,
  -0.03100713901221752,
  -0.060612935572862625,
  0.009260294958949089,
  0.031234554946422577,
  0.13125307857990265,
  -0.012756242416799068,
  0.03591400757431984,
  -0.01191822811961174,
  -0.042370233684778214,
  -0.01862957328557968,
  0.006711157970130444,
  0.030487261712551117,
  0.023814883083105087,
  0.006893133278936148,
  0.0006498689181171358,
  -0.0476195365190506,
  -0.03707505762577057,
  -0.024031586945056915,
  0.013223263435065746,
  0.05725440755486488,
  0.04666541889309883,
  0.019202416762709618,
  -0.02603995054960251,
  0.006440171040594578,
  -0.05567765608429909,
  -0.005431135185062885,
  0.03812488913536072,
  0.07684935629367828,
  0.007395575288683176,
  -0.07512582838535309,
  0.11888226121664047,
  0.018616678193211555,
  -0.006673435680568218,
  0.01798028126358986,
  -0.015800438821315765,
  -0.0017966608284041286,
  0.017662979662418365,
  -0.01641850546002388,
  0.023410649970173836,
  -0.02160937525331974,
  -0.03879554197192192,
  -0.022067788988351822,
  0.026417817920446396,
  0.05402132868766785,
  0.01906670816242695,
  -0.0077096568420529366,
  -0.02210066467523575,
  -0.030961718410253525,
  0.03090527653694153,
  -0.027809979394078255,
  0.038735438138246536,
  -0.03783391788601875,
  -0.01618196815252304,
  0.004230930469930172,
  -0.037576064467430115,
  0.03185654804110527,
  -0.06450198590755463,
  0.0009185735834762454,
  -0.03510133549571037,
  -0.0123573774471879,
  0.010479704476892948,
  0.03813457861542702,
  -0.024355415254831314,
  0.028969615697860718,
  -0.06248248368501663,
  -0.01859181746840477,
  -0.009000178426504135,
  -0.015070803463459015,
  -0.03411509096622467,
  -0.07170674204826355,
  0.017749836668372154,
  0.003982588183134794,
  0.03539244085550308,
  -0.02471897006034851,
  0.0013527754927054048,
  -0.0662342980504036,
  -0.04180503636598587,
  -0.00661375792697072,
  0.02815883420407772,
  0.008304438553750515,
  0.020691094920039177,
  0.03421005606651306,
  -0.0742124393582344,
  0.006539625581353903,
  -0.012376000173389912,
  0.031035447493195534,
  -0.015134613960981369,
  -0.031369827687740326,
  0.04271870478987694,
  -0.050493139773607254,
  0.00894305482506752,
  -0.01609109155833721,
  0.012226845137774944,
  0.01711825467646122,
  0.008857331238687038,
  0.02376426011323929,
  0.0030972708482295275,
  0.08938385546207428,
  -0.006378467660397291,
  0.0027145138010382652,
  0.01633797027170658,
  0.03302580863237381,
  -0.010536123998463154,
  0.01955157145857811,
  0.08490507304668427,
  -0.0703328475356102,
  -0.03926125541329384,
  -4.177086849552351e-33,
  -0.008198140189051628,
  0.029848463833332062,
  -0.01754787750542164,
  0.021348679438233376,
  0.009668402373790741,
  -0.05420871824026108,
  0.005045033525675535,
  0.018147047609090805,
  0.0260432418435812,
  -0.02983391471207142,
  -0.06435056775808334,
  -0.011147461831569672,
  0.011291694827377796,
  0.011635766364634037,
  -0.036793988198041916,
  -0.012345307506620884,
  -0.05683457478880882,
  0.018599189817905426,
  0.012287708930671215,
  -0.011724286712706089,
  0.01856755092740059,
  0.02297106198966503,
  0.03774580731987953,
  0.020020337775349617,
  0.09730570763349533,
  -0.02030770853161812,
  -0.0400434210896492,
  0.06355389207601547,
  -0.010197383351624012,
  0.012913461774587631,
  0.005711055360734463,
  0.01781056821346283,
  0.025951767340302467,
  -0.04219958558678627,
  -0.021590430289506912,
  0.04500625655055046,
  0.042382314801216125,
  -0.0068267337046563625,
  -0.015056813135743141,
  0.003560028038918972,
  -0.05216127261519432,
  -0.0014004469849169254,
  -0.05115736275911331,
  -0.005741954781115055,
  -0.023754984140396118,
  -0.060082122683525085,
  0.024469370022416115,
  -0.008361926302313805,
  -0.008169237524271011,
  -0.007225885055959225,
  0.0011588699417188764,
  -0.003614867338910699,
  -0.007014538627117872,
  0.0028530561830848455,
  0.020203756168484688,
  -0.0552673302590847,
  0.01694289967417717,
  -0.0012949013616889715,
  0.04657363146543503,
  0.05960460752248764,
  0.023164529353380203,
  0.012736218050122261,
  -0.0008500248659402132,
  -0.0575350746512413,
  -0.021595267578959465,
  0.016380541026592255,
  -0.06886734068393707,
  -0.015685277059674263,
  0.04724306985735893,
  -0.002290895441547036,
  0.01989554613828659,
  0.020724555477499962,
  0.022192081436514854,
  0.023035163059830666,
  0.06143036112189293,
  0.030861353501677513,
  -0.039050254970788956,
  -0.0037442955654114485,
  0.035410188138484955,
  0.05702538788318634,
  0.058375827968120575,
  0.05225012078881264,
  -0.026262786239385605,
  -0.04631901904940605,
  -0.012878245674073696,
  -0.016052573919296265,
  -0.05235106125473976,
  0.03680030256509781,
  -0.023203857243061066,
  -0.0023950710892677307,
  0.04674835130572319,
  0.03192664682865143,
  0.007780549116432667,
  0.008432288654148579,
  0.039455898106098175,
  0.01969590038061142,
  0.026227151975035667,
  0.0312839113175869,
  -0.016547875478863716,
  0.0100782485678792,
  0.042250219732522964,
  0.018077101558446884,
  0.05121941491961479,
  -0.018603717908263206,
  0.018363846465945244,
  -0.029833165928721428,
  0.04369720444083214,
  0.005316595081239939,
  0.018910782411694527,
  0.029512127861380577,
  0.018391363322734833,
  0.036682602018117905,
  -0.01730259507894516,
  -0.019905295222997665,
  -0.011457521468400955,
  -0.0032774789724498987,
  -0.00922875851392746,
  -0.062272559851408005,
  0.010189181193709373,
  -0.032376520335674286,
  -0.0034097926691174507,
  0.010533434338867664,
  0.042449090629816055,
  0.005818536039441824,
  0.013801844790577888,
  -0.002950287889689207,
  0.06792780011892319,
  -0.011152681894600391,
  -0.023040814325213432,
  -0.04158736765384674,
  -0.009903251193463802,
  0.007052857428789139,
  1.8670679935439694e-07,
  0.01953800953924656,
  -0.037715278565883636,
  -0.028862757608294487,
  0.05371788516640663,
  -0.052207399159669876,
  0.03694286197423935,
  0.06836878508329391,
  0.0272198673337698,
  -0.020614873617887497,
  0.014903008937835693,
  -0.00885105226188898,
  0.01823948696255684,
  0.04640328511595726,
  -0.015615479089319706,
  -0.02944343164563179,
  0.06939171254634857,
  -0.002278922824189067,
  -0.0168228168040514,
  0.03119172900915146,
  -0.034973807632923126,
  0.06488559395074844,
  -0.04044797271490097,
  0.017964374274015427,
  -0.01755051501095295,
  0.050205837935209274,
  -0.021108951419591904,
  -0.03297041356563568,
  -0.0005428582662716508,
  -0.06126975268125534,
  -0.02880963683128357,
  0.12428528815507889,
  -0.00849155057221651,
  0.005050322040915489,
  0.020559055730700493,
  0.0076638394966721535,
  -0.0718449130654335,
  0.0006847202312201262,
  0.026062553748488426,
  0.0033837640658020973,
  0.019204964861273766,
  0.022912021726369858,
  -0.06168004870414734,
  0.008002996444702148,
  0.056084997951984406,
  -0.009195202961564064,
  0.024758748710155487,
  -0.027370842173695564,
  0.022278474643826485,
  -0.031002214178442955,
  0.02910611592233181,
  -0.016812553629279137,
  0.05458025634288788,
  -0.026967864483594894,
  -0.017533868551254272,
  0.07722625136375427,
  0.03753812983632088,
  -0.016749359667301178,
  0.004386956337839365,
  -0.01288013905286789,
  0.04719328507781029,
  0.057146258652210236,
  -0.0029508438892662525,
  0.01762041449546814,
  0.10666747391223907,
  0.03594304621219635,
  -0.05736410245299339,
  0.04487111046910286,
  1.320520277588766e-34,
  0.012126261368393898,
  -0.011577093973755836,
  0.02863713912665844,
  0.005460101645439863,
  -0.029371093958616257,
  0.0034397176932543516,
  0.023095931857824326,
  -0.016376106068491936,
  0.0464467778801918,
  -0.07012873142957687,
  0.024101359769701958]

In [13]:
# Step 2.1: Connect to your local Weaviate instance
with weaviate.connect_to_local() as client:

    # Step 2.2: Use this collection
    movies = client.collections.use("Euro_Laws")

    # Step 2.3: Perform a vector search with NearVector
    response = movies.query.near_vector(
        near_vector= vector , 
        limit=10
    )

    for obj in response.objects:
        print(json.dumps(obj.properties, indent=10))  # Inspect the results

{
          "subject_matter": "politics and public safety;  European construction;  criminal law;  free movement of capital;  social affairs;  international security",
          "cites": "32011L36; teu_2016/art_5; char_2016; dec_framw/2002/946; tfeu_2016/art_325; 32011L93; 32014L62; 32017L1371; 32009L123; teu_2016/pro_21; tfeu_2016/pro_22; 32013L40; teu_2016/pro_22; dec_framw/2008/841; teu_2016/art_2; 32015L849; tfeu_2016/pro_21; 31997F0625%2801%29; 32014L42; 32002D187; 32008L99; dec_framw/2001/500; dec_framw/2004/757; 32014L57; dec_framw/2009/948; dec_framw/2001/413; dec_framw/2003/568; 32017L541",
          "treaty": "TFEU",
          "status": "In Force",
          "authors": "European Council; European Parliament",
          "celex": "32018L1673",
          "eurovoc": "elimination of terrorism; EU police and customs cooperation; penalty; financial transaction; international criminal law; crime prevention; European security; international crime; European Judicial Network in criminal

In [12]:
!docker run -d \
  -p 8080:8080 \
  -p 50051:50051 \
  -v weaviate_data:/var/lib/weaviate \
  semitechnologies/weaviate:latest


61f168ea797c641ee8a49b94dd8cf8c2c64b74a57fdd47422965b06d47bb9f5b
docker: Error response from daemon: failed to set up container networking: driver failed programming external connectivity on endpoint priceless_grothendieck (be49c4f4f021e0bec29779681883bbd73e237db28f9c124081ed3cc43366724c): Bind for :::8080 failed: port is already allocated

Run 'docker run --help' for more information
